# Summarise Derivative Cover Check Sheets - new format

In [1]:
print('\n\n##############################################')
print('#                                            #')
print('#  START 3/4 derv_checker_summarising.ipynb  #')
print('#                                            #')
print('##############################################\n\n')



##############################################
#                                            #
#  START 3/4 derv_checker_summarising.ipynb  #
#                                            #
##############################################




In [2]:
# Import libraries

import time
start_time          = time.time()
start_time_derv_checker_summarising = start_time
print('Importing libraries to summarise the derivative calcs ...')

import pandas as pd
import re, os, shutil
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from constants import (
    pthEXPORTS,
    pthPy,
    pth_dl,
    pthMandates,
    pthOverdrafts,
    pthSttlmnt,
    pthDaily,
    pthLOCAL,
)
from utilities import timediff, prior_working_day

# create a lookup table for fund UT status and investment team
twoA = pd.read_excel(pthSttlmnt, sheet_name = 'Funds', usecols = 'A, D:E')

print(f' {timediff(start_time, time.time())} importing libraries to summarise the derivative calcs', '\n')

Importing libraries to summarise the derivative calcs ...
 6.9sec importing libraries to summarise the derivative calcs 



In [3]:
# Get report date and selected summary sheet option

start_time = time.time()
print('Getting the reporting date and names of completed derivative files ...')

df      = pd.read_excel(pthPy, sheet_name="arc", header=None, usecols="A,E").dropna(subset = [0])
k       = df.iloc[2,1]
rptDate = k if isinstance(k, datetime) else prior_working_day(datetime.today()) # prior working day or report date override; has type datetime()
summ_yn = df.iloc[3, 1]
full    = df[0].iloc[1:]
funds   = (",").join(full.tolist())

# get fund names of current derivative calculation files
y       = os.scandir(pthEXPORTS)
pattern = f'Derv Calc {rptDate.strftime("%d%b%Y")}.xlsx'
start_y = time.time()
print('', f'Get scandir() of derivative calcs folder for {rptDate.strftime("%a %d %B %Y")}')
funds   = []
for s in tqdm(y):
    if s.name.endswith(pattern): # all 'XXXX Derv  Calc ddMmmYYYY.xlsx' files
        funds.append(s.name[:-25]) # fund codes of all files with current report dates
print('', f'{len(funds)} funds')
print('', f'{timediff(start_y, time.time())} to get scandir() of derivative calcs folder for {rptDate.strftime("%a %d %B %Y")}')

print(f' {rptDate.strftime("%A %d %b %Y")} for {len(full)} funds: {(", ").join(full.tolist())}\n')
print(f' {"No" if summ_yn == "No" else "A"} summary sheet is required')

print(f' {len(funds)} completed derivative calculation files at {rptDate.strftime("%A %#d %B %Y")}')
print(f'\n {timediff(start_time, time.time())} getting the reporting date and names of completed derivative files\n')

Getting the reporting date and names of completed derivative files ...
 Get scandir() of derivative calcs folder for Mon 01 June 2026


45414it [00:00, 478944.66it/s]

 151 funds
 0.1sec to get scandir() of derivative calcs folder for Mon 01 June 2026
 Monday 01 Jun 2026 for 151 funds: 3BBCIINC, ABMMBND, ADRRC, ADVMB, AFGBAL, AFLBAL, AFLMMF, AMMARF, AMPBQP, ASBTOS, ASHFLX, BCIFIF, BPROV, CCNPF, CMPFCASH, CMPFFLEX, CMPFINC, CSIRBQP, ECICBALC, ELCIPF, ENGENIP, ENGENMBF, FEMPBF, GACASH, GAEMBF, GEMSMEDC, GMRETF, GMRETF2, GRFINV, GTCWP2, HOLADC, HOLDINC, HOLEQU, HOLIDC, HOLMMF, HOLYPF, HOSMED, IJGCOR, IJGIPF, IMPALA, IMPBAL_C, IMPREF_C, IPIPF, ISPFP, LEZAFFI, LEZALDI, LPIIFTAA, MASAINC, MASAINRF, MASAMMF, MASASI, MASASIRF, MEDINC, MGFYQP, MOMBBF, MOMBBRF, MOMFLX, MOMFXB, MOMIPF, MOMPRET, MOMTAAHI, MOMTAALI, MOMTAAMI, MULTICH, MWPFEQU, MWPFILB, MYQIP, NEDMED, NESEQU, NFMWEQU, NFMWAGG, NGKINC, OMMAIF, PABS, PBFETF, PBNDQ, PCBF, PCCEF, PCEQTF, PCGEARF, PCGEF, PCSHQ, PEQ, PEQF, PETFIP, PEYF, PFFIF, PGCBF, PGCEF, PGIPFA, PGPCEM_C, PGPCGE_C, PGPGARF, PGPGBF_C, PGPGIF_C, PGPRF_C, PICPROV, PIF, PIMBAL, PIMEVO, PIMIDF, PIPF, PIPFP, PLMED, PLPRNA, PMMF, POIF_C, PO

In [17]:
# dataframe the fund derivative calc summaries

start_time = time.time()
print(f'Populating the summary dataframe with {len(funds)} funds for {rptDate.strftime("%A %#d %B %Y")} ...')

# create a dataframe with the necessary columns
summary = pd.DataFrame(funds, columns = ['Fund Code'])

cols = ['UT?', '#', 'Cash Cover', 'Cash Cover 2', 'Incl CLNs & longer-dated debt', 'Incl Underlying UTs', 'SA Equity Indices', \
        'Ex-SA Equity Indices', 'SA Bond Indices', 'Ex-SA Bond Indices', 'Currency Futures', 'Currency Forwards', 'Swaps', 'FRAs', \
        'SA Bond Cover', 'Foreign Equity Cover', 'Local Equity Cover', 'Total Equity', 'Total Foreign', 'Global Exposure', \
        'Net Effective Exposure', 'Fund Rules', 'Team', 'PIM Overdrafts', 'Leverage (Gross)', 'Leverage (Net)']

for col in cols: # create an empty summary dataframe with the given column headings
    summary[col] = ''
    # https://www.reddit.com/r/learnpython/comments/n1ee17/how_to_add_multiple_empty_columns_into_my_data/

# read each fund's 'xxxx Derv Calc ddmmmyyyy.xlsx' sheet into the summary dataframe
if summ_yn != 'No':
    for index, fund in enumerate(tqdm(funds)): # https://stackoverflow.com/questions/522563/how-to-access-the-index-value-in-a-for-loop
        #wb_f  =  xl.Workbooks.Open(pthEXPORTS + fr'\{fund} Derv Calc {rptDate}.xlsx') # open the calc sheet, allowing it to update cell values
        #wb_f.Close(SaveChanges = True)                                                # save and close the calc sheet 
        fn    = pthEXPORTS + fr'\{fund}' + f' Derv Calc {rptDate.strftime("%d%b%Y")}.xlsx'     # access each fund's 'xxxx Derv Calc ddmmmyyyy.xlsx' sheet
        ddfSm = pd.read_excel(fn, header = None, sheet_name = 'Summary'   ) # access data on the Summary sheet
        # ddfGE = pd.read_excel(fn, header = None, sheet_name = 'Glbl Expsr') # access data on the Global Exposure Sheet
        # summary.iat[index,  1] = twoA[twoA['Fund Code'] == fund].iloc[0, 2] # column 'B' of summary sheet = "G3"  'UT' or '≠UT'
        summary.iat[index,  1] = twoA[twoA.iloc[:,0] == fund].iloc[0, 2] # column 'B' of summary sheet = "G3", has value 'UT' or '≠UT'        
        summary.iat[index,  2] = ddfSm.iat[ 0, 4].astype(int)      # 'E1'  'number of derivatives'
        summary.iat[index,  3] = round(ddfSm.iat[37, 2]      , 3)  # 'C38' 'In/Adequate derivative cover for derivatives'
        summary.iat[index,  4] = round(ddfSm.iat[37, 4]      , 3)  # column 'E' of summary sheet = cash cover 2
        summary.iat[index,  5] = round(ddfSm.iat[37, 2]      , 3)  # column 'F' of summary sheet = 'C10'
        summary.iat[index,  6] = round(ddfSm.iat[35, 2]      , 3)  # column 'G' of summary sheet = 'C34' 'Cash from the underlying UTs'
        summary.iat[index,  7] = round(ddfSm.iat[ 1, 7] * 100, 3)  # column 'H' of summary sheet = 'H2' SA equity indices
        summary.iat[index,  8] = round(ddfSm.iat[ 4, 7] * 100, 3)
        summary.iat[index,  9] = round(ddfSm.iat[ 7, 7] * 100, 3)
        summary.iat[index, 10] = round(ddfSm.iat[10, 7] * 100, 3)       
        summary.iat[index, 11] = round(ddfSm.iat[13, 7] * 100, 3)
        summary.iat[index, 12] = round(ddfSm.iat[16, 7] * 100, 3)
        summary.iat[index, 13] = round(ddfSm.iat[19, 7] * 100, 3)  # "H20" 'Swaps'
        summary.iat[index, 14] = round(ddfSm.iat[22, 7] * 100, 3)  # 'H23' 'FRAs'
        summary.iat[index, 15] = round(ddfSm.iat[ 5, 4]      , 3)  # "E6"  'SA bond cover'
        summary.iat[index, 16] = round(ddfSm.iat[ 9, 4]      , 3)  # "E10" 'Foreign equity cover'
        summary.iat[index, 17] = round(ddfSm.iat[13, 4]      , 3)  # "E14" 'Local equity cover'
        # summary.iat[index, 18] = round(ddfSm.iat[44, 2] * 100, 3)  # "C45" 'Total equity, incl property equity'
        # summary.iat[index, 19] = round(ddfSm.iat[58, 2]      , 3)  # "C59" 'Total foreign'
        # summary.iat[index, 20] = round(ddfGE.iat[ 1, 3] * 100, 3)
        # summary.iat[index, 21] = round(ddfGE.iat[ 2, 3] * 100, 3)
        summary.iat[index, 22] = fund                              # f'{fund} calc sheet'
        # summary.iat[index, 23] = twoA.loc[twoA['Fund Code'] == f'{fund}'].iat[0,1] # investment team lookup on 2AXX sheet
        summary.iat[index, 23] = twoA[twoA.iloc[:,0] == fund].iloc[0, 1] # investment team lookup on 2AXX sheet        
        summary.iat[index, 24] = round(ddfSm.iat[ 3, 1], 2)  # fund NAV
        summary.iat[index, 25] = round(ddfSm.iat[18, 4], 1)  # fund leverage (gross) #####
        summary.iat[index, 26] = round(ddfSm.iat[19, 4], 1)  # fund leverage (net)
        # Using at[] and iat[] instead of loc[] and iloc[]
        # https://stackoverflow.com/questions/28757389/pandas-loc-vs-iloc-vs-at-vs-iat
        # at and iat are meant to access a scalar, that is, a single element in the dataframe, 
        # while loc and iloc are meant to access several elements at the same time, 
        # potentially to perform vectorized operations
        # https://medium.com/codex/dont-use-loc-iloc-with-loops-in-python-instead-use-this-f9243289dde7
    
        # sort and reindex the summary dataframe
        # https://stackoverflow.com/questions/17141558/how-to-sort-a-pandas-dataframe-by-two-or-more-columns
        # https://stackoverflow.com/questions/33165734/update-index-after-sorting-data-frame
        summary.reset_index(inplace = True, drop = True)
        # summary.sort_values(by = ['Cash Cover', '#'], ascending = [True, False], ignore_index = True) # inplace = True, )
        # df = df.reset_index().sort_values(by=['Date', 'index']).drop(['index'], axis=1)
        # https://stackoverflow.com/questions/48066933/pandas-sorting-days-whilst-preserving-order

print(f' {timediff(start_time, time.time())} populating the summary dataframe with {len(funds)} funds for {rptDate.strftime("%A %#d %B %Y")}\n')

summary = summary.sort_values(by = 'Cash Cover', ascending = True) # sort the cover calc dataframe by 'Cash Cover' in ascending order
# summary

Populating the summary dataframe with 151 funds for Monday 1 June 2026 ...


100%|████████████████████████████████████████████████████████████████████████████████| 151/151 [00:23<00:00,  6.54it/s]

 23.1sec populating the summary dataframe with 151 funds for Monday 1 June 2026



In [18]:
#summary

,Fund Code,UT?,#,Cash Cover,Cash Cover 2,Incl CLNs & longer-dated debt,Incl Underlying UTs,SA Equity Indices,Ex-SA Equity Indices,SA Bond Indices,...,Local Equity Cover,Total Equity,Total Foreign,Global Exposure,Net Effective Exposure,Fund Rules,Team,PIM Overdrafts,Leverage (Gross),Leverage (Net)
46,LPIIFTAA,TAA,7,-15.353,NaN,-15.353,21.774,0.732,NaN,NaN,...,0.0,,,,,LPIIFTAA,MultAsst,105650675.31,120.1,64.6
43,ISPFP,UT,0,0.084,NaN,0.084,0,NaN,NaN,NaN,...,0.0,,,,,ISPFP,Bonds,793658575.18,99.9,99.9
134,SMMIBF,UT,0,0.176,NaN,0.176,0,NaN,NaN,NaN,...,0.0,,,,,SMMIBF,Bonds,2298258124.33,99.8,99.8
83,PEQF,UT,2,0.342,NaN,0.342,0,29.324,NaN,NaN,...,0.0,,,,,PEQF,Eqty,1990690764.91,99.7,99.7
32,HOLEQU,UT,1,0.345,NaN,0.345,0,8.856,NaN,NaN,...,0.0,,,,,HOLEQU,Eqty,285932612.24,99.7,99.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,CSIRBQP,≠UT,0,100,NaN,100,0,NaN,NaN,NaN,...,0.0,,,,,CSIRBQP,Bonds,230260127.23,99.9,99.9
16,CMPFINC,≠UT,0,100,NaN,100,0,NaN,NaN,NaN,...,0.0,,,,,CMPFINC,FxdInt,352703869.34,84.3,84.3
45,LEZALDI,≠UT,0,100,NaN,100,0,NaN,NaN,NaN,...,0.0,,,,,LEZALDI,Bonds,397129827.58,43.8,43.8
150,VMPTAA_C,TAA,3,100.0,NaN,100.0,77.276,-21.362,NaN,NaN,...,-0.214,,,,,VMPTAA_C,MultAsst,48111104.61,137.3,55.9


In [19]:
# Get report date and selected summary sheet option and then populate the summaries in a dataframe

start_time = time.time()
print(f'Sorting and then saving the summary dataframe with {len(funds)} funds for {rptDate.strftime("%A %#d %B %Y")} ...')

# sort the summary dataframe and save it to a new Excel file
# ut_types = ['UT', '≠UT', 'UCITS', 'SAA', 'TAA', 'ICAV']
df1 = pd.read_excel(pthSttlmnt, sheet_name="Funds", usecols="E").dropna()
ut_types = df1.iloc[:,0].unique().tolist()
sorted_summary = pd.DataFrame([]) # empty dataframe
for ut_type in ut_types: # stack the > 0 derivative funds first ...
    summary_subset = summary[(summary['UT?'] == ut_type) & (summary['#'] != 0)]
    sorted_summary = pd.concat([sorted_summary, summary_subset])
    # sorted_summary = sorted_summary.sort_values(by = 'Cash Cover', ascending = False) # sort the summary dataframe

for ut_type in ut_types: # ... then stack the no derivative funds
    summary_subset = summary[(summary['UT?'] == ut_type) & (summary['#'] == 0)]
    sorted_summary = pd.concat([sorted_summary, summary_subset])
    # sorted_summary = sorted_summary.sort_values(by = 'Cash Cover', ascending = False) # sort the summary dataframe

sorted_summary.reset_index(inplace = True, drop = True)

print(f' {timediff(start_time, time.time())} sorting and then saving the summary \
dataframe with {len(funds)} funds for {rptDate.strftime("%A %#d %B %Y")}\n')

Sorting and then saving the summary dataframe with 151 funds for Monday 1 June 2026 ...
 0.3sec sorting and then saving the summary dataframe with 151 funds for Monday 1 June 2026



In [21]:
sorted_summary

,Fund Code,UT?,#,Cash Cover,Cash Cover 2,Incl CLNs & longer-dated debt,Incl Underlying UTs,SA Equity Indices,Ex-SA Equity Indices,SA Bond Indices,...,Local Equity Cover,Total Equity,Total Foreign,Global Exposure,Net Effective Exposure,Fund Rules,Team,PIM Overdrafts,Leverage (Gross),Leverage (Net)
0,PEQF,UT,2,0.342,NaN,0.342,0,29.324,NaN,NaN,...,0.0,,,,,PEQF,Eqty,1990690764.91,99.7,99.7
1,HOLEQU,UT,1,0.345,NaN,0.345,0,8.856,NaN,NaN,...,0.0,,,,,HOLEQU,Eqty,285932612.24,99.7,99.7
2,PEQ,UT,1,0.587,NaN,0.587,0,16.946,NaN,NaN,...,0.0,,,,,PEQ,Eqty,252853069.23,99.4,99.4
3,SMMRRF,UT,1,0.825,NaN,0.825,0,23.777,NaN,NaN,...,0.0,,,,,SMMRRF,MultAsst,711570443.6,99.2,99.2
4,ADRRC,UT,9,1.515,NaN,1.515,0,18.23,NaN,NaN,...,0.0,,,,,ADRRC,MultAsst,507653122.95,170.0,98.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,CMPFINC,≠UT,0,100,NaN,100,0,NaN,NaN,NaN,...,0.0,,,,,CMPFINC,FxdInt,352703869.34,84.3,84.3
147,LEZALDI,≠UT,0,100,NaN,100,0,NaN,NaN,NaN,...,0.0,,,,,LEZALDI,Bonds,397129827.58,43.8,43.8
148,MOMFXB,≠UT,0,118.594,NaN,118.594,18.67,NaN,NaN,NaN,...,0.0,,,,,MOMFXB,Bonds,3717788515.91,155.3,118.1
149,UWRFCON,TAA,0,100,NaN,100,0,NaN,NaN,NaN,...,0.0,,,,,UWRFCON,MultAsst,6509416.1,0.0,0.0


In [8]:
start_time = time.time()
print('Writing the dataframe to a sheet ...')

# TEST ++++++++++++++++

# # save the summary dataframe to a new Excel file as 'Derv ddmmyyyy.xlsx'
# sorted_summary.to_excel(pthEXPORTS + f'\Derv {rptDate}.xlsx', index = False, sheet_name = 'Summary')
# print(' ', pthEXPORTS + f'\Derv {rptDate}.xlsx')

# dataframe the PARN and Derv reports from the local Downloads folder
fPARN   = os.path.join(pth_dl, f"PARN ({len(full)}) {rptDate.strftime('%d%b%Y')}.csv")
wbH     = pd.read_csv(fPARN)
fDE     = os.path.join(pth_dl, f"DERV ({len(full)}) {rptDate.strftime('%d%b%Y')}.csv")
wbD     = pd.read_csv(fDE)

# rename 'UNKNOWNs' as 'SWAPS' where they are not SYTH or empty portfolio holdings
# https://stackoverflow.com/questions/36909977/update-row-values-where-certain-condition-is-met-in-pandas
unknowns_filter = \
  (wbH['Valuation First Level'] == 'UNKNOWN') &\
  (wbH['Sub Security Type']     == 'TRS') &\
  (wbH['Investment Type']       != 'SYTH') &\
  pd.notna(wbH['Investment Type'])
wbH.loc[unknowns_filter, ['Valuation First Level', 'Valuation Second Level']] = 'SWAPS'
unknowns = wbH.loc[unknowns_filter]
print(f'  {len(unknowns)} "UNKNOWN" securities found and amended: {(", ").join(unknowns["PrimaryAssetID"].tolist())}')

# identify "No Data found for this Entity" funds
no_data_filter = (wbH['i Issue Name'] == 'No Data found for this Entity')
no_data = wbH.loc[no_data_filter]
print(f'  {len(no_data)} "No data" fund{"s" if len(no_data["Entity ID"]) != 1 else ""}: {(", ").join(no_data["Entity ID"].tolist())}')

# convert date columns to datetime format
date_cols = ['i Position Effective Date', 'Maturity Date', 'Next Coupon Date']
for date_col in date_cols:
    wbH[date_col] = pd.to_datetime(wbH[date_col])

# convert holdings numerical columns to numbers
num_cols = ['Original Nominal','Clean Book Value','Clean Market Value','Accrued Income','Dividend Receivable',
     'Sum of Market Value Income','Market Price /Yield','% of Total Market Value','Coupon','Duration',
     'Modified Duration','NACA Yield','NACM Yield','Weighted Avg NACA Yield','Weighted Avg NACM Yield',
     'Market Value %','Current Exposure','Current Exposure %','Weighted Average NACS Yield',
     'Weighted Average Coupon','Weighted Modified Duration']
for num_col in num_cols:
    wbH[num_col] = wbH[num_col].astype(str).str.replace(',', '').astype(float)

# convert deltas numerical columns to numbers
num_cols_dervs = ['Nominal Holding','Delta','Market Value','Effective Exposure']
for num_col_derv in num_cols_dervs:
    wbD[num_col_derv] = wbD[num_col_derv].astype(str).str.replace(',', '').astype(float)

# write the summary, fund holdings, and derivative deltas to a workbook
summary_name = pthEXPORTS + f'\Derv {rptDate.strftime("%d%b%Y")}.xlsx'  # assign the file name
writer       = pd.ExcelWriter(summary_name, engine = 'xlsxwriter')      # instantiate a sheet writer with file name
sorted_summary.to_excel(writer, index = False, sheet_name = 'Summary')  # write the summary sheet
wbH.to_excel(           writer, index = False, sheet_name = 'Holdings') # write the fund holdings sheet
wbD.to_excel(           writer, index = False, sheet_name = 'Derv')     # write the derivative deltas sheet
writer.close() # https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html   class for writing DataFrame objects into excel sheets

print(f"\nSummary file:\n {summary_name}\n")
print(f'{timediff(start_time, time.time())} writing the dataframe to a sheet')

# sorted_summary

Writing the dataframe to a sheet ...
  0 "UNKNOWN" securities found and amended: 
  0 "No data" funds: 

Summary file:
 \\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Derivative Cover\Derv 01Jun2026.xlsx

9.8sec writing the dataframe to a sheet


In [9]:
# prettify the summary sheet and add hyperlinks with xlwings

print(f'Prettifying and adding links to the summary sheet with xlwings ...')
start_time = time.time()

import xlwings as xw
if summ_yn != 'No':
    wbS    = xw.Book(pthEXPORTS + f'\Derv {rptDate.strftime("%d%b%Y")}.xlsx')
    shtS   = wbS.sheets['Summary']                         # derivative cover summary sheet
    #xl.DisplayAlerts = False                              # suppress Excel warning dialogues
    
    # format the headings row of the Summary file
    shtS['A1'].add_hyperlink(pthEXPORTS,f'Derivative Cover Calcs {rptDate.strftime("%d%b%Y")}')
    shtS.range('A:A').column_width = 20.71
    shtS.range('B:B').column_width =  5.29
    shtS['W1'].add_hyperlink(r'P:\Investment Operations\GRC\Compliance\Client Mandates', 'Fund Mandate')
    shtS.range('W:W').column_width = 13.71
    shtS['X1'].add_hyperlink(pthSttlmnt,'Team')
    shtS.range('X:X').column_width = 11.71
    shtS['Y1'].value = 'NAV'
    shtS['Y:Y'].number_format = "#,##0.00" # https://stackoverflow.com/questions/55391542/adjust-number-formatting-in-excel-via-xlwings-from-python
    shtS['Z1'].add_hyperlink(pthOverdrafts,'PIM Overdrafts')
    shtS['Z1'].api.HorizontalAlignment = -4108 # xlCenter
    shtS['Z1'].api.VerticalAlignment   = -4160  # xlTop
    shtS['A1:AA1'].api.WrapText = True # https://docs.xlwings.org/en/stable/missing_features.html
    shtS['A1:AA1'].font.bold = True # https://docs.xlwings.org/en/stable/missing_features.html
    
    # add investment team names and links to fund mandates and calculation sheets
    start_time_links = time.time()
    print(' ', 'Adding investment team names and links to fund mandates and calculation sheets')
    for index, row in tqdm(sorted_summary[['Fund Code']].iterrows(), total = sorted_summary[['Fund Code']].shape[0]): # iterate over the funds        
        #shtS['A' + str(index + 2)].value = f'{shtS["A" + str(index + 2)]} calc sheet'                                         # fund code
        shtS['A' + str(index + 2)].add_hyperlink(fr'{pthEXPORTS}\{row[0]} Derv Calc {rptDate.strftime("%d%b%Y")}.xlsx', f'{row[0]}') # link to calc sheet
        shtS['W' + str(index + 2)].add_hyperlink(fr'{pthMandates}\{row[0]} Rules.docx',f'{row[0]}')                            # link to fund mandate
    print(' ', f'{timediff(start_time_links, time.time())} adding investment team names and links to fund mandates and calculation sheets')
    
    # add conditional formating for values that are negative or exceed 100% of NAV
    start_time_format = time.time()
    print('\n', ' ', f'Adding conditional formats for values that are negative or exceed 100% of NAV; {len(funds) * 19} = {len(funds)} funds x 19 columns')
    for a_cell in tqdm(shtS['D2:V2'].expand('down')):
        if type(a_cell.value) in [float, int]:
            if a_cell.value < 0:
                a_cell.font.color = (255,   0,   0) # red is (255,0,0) in RGB or #FF0000 in Hex  
            if a_cell.value > 100 or a_cell.value < -100:
                a_cell.color      = (255, 197, 255) # light pink for cell colour
    print(' ', f'{timediff(start_time_format, time.time())} adding conditional formats for values that are negative or exceed 100% of NAV')

if summ_yn != 'No':
    wbS.save()
    wbS.close()

print(' ', pthEXPORTS + f'\Derv {rptDate.strftime("%d%b%Y")}.xlsx')
print(f'\n{timediff(start_time, time.time())} prettifying and adding links to the summary sheet with xlwings\n')

Prettifying and adding links to the summary sheet with xlwings ...
  Adding investment team names and links to fund mandates and calculation sheets


  0%|                                                                                          | 0/151 [00:00<?, ?it/s]C:\Users\hilton.netta\AppData\Local\Temp\ipykernel_30584\928037208.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  shtS['A' + str(index + 2)].add_hyperlink(fr'{pthEXPORTS}\{row[0]} Derv Calc {rptDate.strftime("%d%b%Y")}.xlsx', f'{row[0]}') # link to calc sheet
C:\Users\hilton.netta\AppData\Local\Temp\ipykernel_30584\928037208.py:34: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  shtS['W' + str(index + 2)].add_hyperlink(fr'{pthMandates}\{row[0]} Rules.docx',f'{row[0]}')                            

  6.9sec adding investment team names and links to fund mandates and calculation sheets

   Adding conditional formats for values that are negative or exceed 100% of NAV; 2869 = 151 funds x 19 columns


100%|██████████████████████████████████████████████████████████████████████████████| 2869/2869 [01:11<00:00, 40.39it/s]


  1min 11.1sec adding conditional formats for values that are negative or exceed 100% of NAV
  \\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Derivative Cover\Derv 01Jun2026.xlsx

1min 35.7sec prettifying and adding links to the summary sheet with xlwings



In [10]:
# Save the summary dataframe to the derv_summary.xlsx template using xlwings

start_time = time.time()
print(f'Saving the summary dataframe to the derv_summary.xlsx template; {(len(funds) + 1) * 19} = ({len(funds)} funds + 1) x 19 columns')

# open the derv_summary.xlsx derv template and assign values to the holdings and deltas sheets
import xlwings as xw
with xw.App(visible=False) as app:
    # populate the summary sheet with derivative cover calc values for each fund
    wb     = xw.Book(os.path.join(pthDaily, 'derv_summary.xlsx')) # open the derv calc workbook as an object
    shtS_S = wb.sheets['Summary']                                # assign sheet containing the funds derivative cover summary
    shtS_S.clear()                                               # clear the receiving holdings sheet
    shtS_S.range('A1').options(index = False).value = sorted_summary    # paste fund holdings

    # conditional formatting    https://stackoverflow.com/questions/72374261/xlwings-conditional-formatting-based-on-value
    for a_cell in tqdm(shtS_S["D1:V1"].expand("down")):
        if type(a_cell.value) in [float, int]:   # ensure the cell bveing formatted is a float or an integer
            if abs(a_cell.value) >= 100:
                a_cell.color = (255, 204, 255)   # (255, 204, 255) or #FFCCFF is light pink
            elif a_cell.value < 0:
                a_cell.font.color = (255, 0, 0)  # (255,   0,   0) or #FF0000 is red

    # add fund derv calc hyperlinks
    for a_cell in shtS_S["A2:A2"].expand("down"):
        a_cell.add_hyperlink(os.path.join(pthEXPORTS, f'{a_cell.value} Derv Calc {rptDate.strftime("%d%b%Y")}.xlsx'), a_cell.value, screen_tip=None)

    # add fund mandate hyperlinks
    for a_cell in shtS_S["W2:W2"].expand("down"):
        a_cell.add_hyperlink(os.path.join(pthMandates, f'{a_cell.value} rules.docx'), a_cell.value, screen_tip=None)

    # add heading hyperlinks
    shtS_S["A1"].add_hyperlink(pthEXPORTS, f'{rptDate.strftime("%a %d %b %Y")} Calcs', screen_tip=None)
    shtS_S["W1"].add_hyperlink(pthMandates  , "Fund Rules"    , screen_tip=None)
    shtS_S["Y1"].add_hyperlink(pthOverdrafts, "PIM Overdrafts", screen_tip=None)

    # make hyperlinked headings bold
    shtS_S["A1"].font.bold = True
    shtS_S["W1"].font.bold = True
    shtS_S["Y1"].font.bold = True
    
    # wrap text
    shtS_S["A1"].WrapText = True
    shtS_S["W1"].WrapText = True
    shtS_S["Y1"].WrapText = True
    
    # save the summary sheet and then close it
    wb.save(os.path.join(pthDaily, 'derv_summary.xlsx'))   # save the file
    wb.close()

print(' ', os.path.join(pthDaily, 'derv_summary.xlsx'))

print(f'\n {timediff(start_time, time.time())} saving the summary dataframe to the derv_summary.xlsx template\n')

# P:\Investment Operations\GRC\Compliance\Daily\derv_summary.xlsx

Saving the summary dataframe to the derv_summary.xlsx template; 2888 = (151 funds + 1) x 19 columns


100%|██████████████████████████████████████████████████████████████████████████████| 2888/2888 [00:51<00:00, 55.60it/s]


  \\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\Daily\derv_summary.xlsx

 1min 16.6sec saving the summary dataframe to the derv_summary.xlsx template



In [11]:
# Delete contents of the temporary local folder

print(f'Deleting contents of the local temporary folder ...')
start_time = time.time()

local_folder_delete = 'yes'
if local_folder_delete == 'yes':
    for filename in tqdm(os.listdir(pthLOCAL)):    
        file_path = os.path.join(pthLOCAL, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f"Couldn't delete {file_path} because {e}")
        
print(f'{timediff(start_time, time.time())} deleting contents of the local temporary folder completed \n')

print(f'{timediff(start_time_derv_checker_summarising, time.time())} roundtrip time to summarise derivative calcs \n')

Deleting contents of the local temporary folder ...


0it [00:00, ?it/s]

0.0sec deleting contents of the local temporary folder completed 

4min 24.1sec roundtrip time to summarise derivative calcs 



In [12]:
# derv calc template    P:\Investment Operations\GRC\Compliance\Daily\derv.xlsx
# derv summary template P:\Investment Operations\GRC\Compliance\Daily\derv_summary.xlsx
# exports folder        P:\Investment Operations\GRC\Compliance\Derivative Cover

In [13]:
print('\n\n##############################################')
print('#                                            #')
print('#   END 3/4 derv_checker_summarising.ipynb   #')
print('#                                            #')
print('##############################################\n\n')



##############################################
#                                            #
#   END 3/4 derv_checker_summarising.ipynb   #
#                                            #
##############################################


